# Tumulus LiDAR Detector

Detects burial mounds (tumuli) in Romania's 0.5 m airborne LiDAR, in your browser. No install.

## To start: open the Runtime menu, click Run all, and wait about a minute.

The interactive map and the results are generated by the code, so they appear only after you run. The blue area below is where the model can scan (0.5 m LiDAR, ANCPI LAKI III; mostly Oltenia and SW Romania).

<img src="https://raw.githubusercontent.com/ObuObuHub/tumulus-lidar-detector/main/assets/coverage_preview.png" width="460">

After it runs, click inside the blue on the map (or drag the pin) to scan elsewhere, then re-run the Scan cell. Form is not proof: only fieldwork confirms a tumulus. Repo: https://github.com/ObuObuHub/tumulus-lidar-detector

In [ ]:
#@title Setup (run once) { display-mode: "form" }
import os
if not os.path.isdir('tumulus-lidar-detector') and os.path.basename(os.getcwd()) != 'tumulus-lidar-detector':
    !git clone -q https://github.com/ObuObuHub/tumulus-lidar-detector.git
if os.path.basename(os.getcwd()) != 'tumulus-lidar-detector':
    %cd tumulus-lidar-detector
!pip install -q pyproj ipyleaflet
print("Ready. Use the map below.")

In [ ]:
#@title Map - click inside the blue to place your pin (or drag it) { display-mode: "form" }
import json
from IPython.display import display

PIN = {'lat': 44.043, 'lon': 23.522}   # default: a known mound field, inside coverage
_g = {'busy': False}                    # re-entrancy guard for the two-way sync

try:
    from ipywidgets import FloatText, HBox, HTML
    lat_box = FloatText(value=PIN['lat'], description='lat:', step=0.0001)
    lon_box = FloatText(value=PIN['lon'], description='lon:', step=0.0001)
    _info = HTML()
    _have_widgets = True
except Exception as _e:
    _have_widgets = False
    print("Widgets unavailable (%s); set coordinates in the Scan cell." % _e)

def _set_label():
    if _have_widgets:
        _info.value = ("Pin: lat <code>%.5f</code>, lon <code>%.5f</code>"
                       " &nbsp;-&nbsp; now run the Scan cell." % (PIN['lat'], PIN['lon']))

_map_ok = False
if _have_widgets:
    try:
        from google.colab import output as _co
        _co.enable_custom_widget_manager()
    except Exception:
        pass
    try:
        from ipyleaflet import Map, GeoJSON, Marker, basemaps
        _cov = json.load(open('assets/laki3_coverage.geojson'))
        _m = Map(center=(44.7, 23.3), zoom=7, scroll_wheel_zoom=True,
                 basemap=basemaps.OpenStreetMap.Mapnik)
        _m.add(GeoJSON(data=_cov, name='0.5 m LiDAR coverage',
                       style={'color': '#1d4ed8', 'fillColor': '#2563eb',
                              'fillOpacity': 0.30, 'weight': 1}))
        _mk = Marker(location=(PIN['lat'], PIN['lon']), draggable=True, title='Scan here')
        _m.add(_mk)

        def _boxes_from_pin():
            _g['busy'] = True
            lat_box.value, lon_box.value = PIN['lat'], PIN['lon']
            _g['busy'] = False
            _set_label()

        def _on_click(**kw):
            if kw.get('type') == 'click':
                la, lo = kw['coordinates']
                PIN['lat'], PIN['lon'] = round(la, 5), round(lo, 5)
                _g['busy'] = True; _mk.location = (PIN['lat'], PIN['lon']); _g['busy'] = False
                _boxes_from_pin()
        _m.on_interaction(_on_click)

        def _on_drag(_change):
            if _g['busy']:
                return
            PIN['lat'], PIN['lon'] = round(_mk.location[0], 5), round(_mk.location[1], 5)
            _boxes_from_pin()
        _mk.observe(_on_drag, 'location')
        _map_ok = True
    except Exception as _e:
        print("Map unavailable (%s); use the lat/lon boxes below." % _e)

if _have_widgets:
    def _on_box(_change):
        if _g['busy']:
            return
        PIN['lat'], PIN['lon'] = float(lat_box.value), float(lon_box.value)
        if _map_ok:
            _g['busy'] = True; _mk.location = (PIN['lat'], PIN['lon']); _g['busy'] = False
        _set_label()
    lat_box.observe(_on_box, 'value'); lon_box.observe(_on_box, 'value')
    _set_label()
    if _map_ok:
        display(_m)
    display(HBox([lat_box, lon_box]), _info)

In [ ]:
#@title Scan the pinned area { display-mode: "form" }
area_km = 1  #@param {type:"slider", min:1, max:5, step:1}

import os, json
# clear any previous run's outputs so a new scan can never show stale results
for _f in ('/tmp/zone_dets.csv', 'review/zone_view.jpg', 'review/zone_board.jpg', 'detected_mounds.csv'):
    if os.path.exists(_f):
        os.remove(_f)

try:
    lon, lat = PIN['lon'], PIN['lat']
except NameError:
    lat, lon = 44.043, 23.522

def _inside(lon, lat):
    cov = json.load(open('assets/laki3_coverage.geojson'))
    for f in cov['features']:
        ring = f['geometry']['coordinates'][0]
        c = False; n = len(ring); j = n - 1
        for i in range(n):
            xi, yi = ring[i]; xj, yj = ring[j]
            if ((yi > lat) != (yj > lat)) and (lon < (xj - xi) * (lat - yi) / (yj - yi) + xi):
                c = not c
            j = i
        if c:
            return True
    return False

print("Scan point: lat %s, lon %s  |  box %s km" % (lat, lon, area_km))
if not _inside(lon, lat):
    print("\nThis point is OUTSIDE the 0.5 m LiDAR coverage (the blue area on the map).")
    print("Move the pin into the blue and run this cell again.")
else:
    !python tools/scan_zone.py {lon} {lat} {area_km}

In [ ]:
#@title Results { display-mode: "form" }
import os, pandas as pd
from IPython.display import Image, HTML, display

if not os.path.exists('review/zone_view.jpg'):
    print("No result yet. Run the Scan cell above on a point inside the blue coverage.")
else:
    n = 0
    if os.path.exists('/tmp/zone_dets.csv'):
        det = pd.read_csv('/tmp/zone_dets.csv')
        kept = det[det['keep'] == 1] if 'keep' in det.columns else det.iloc[0:0]
        n = len(kept)
    if n > 0:
        print("Found %d probable mound(s). Coordinates and CSV below; green circles on the map." % n)
    else:
        print("No mounds found in this area. This is normal for most points: the model is selective.")
    display(Image('review/zone_view.jpg'))
    if os.path.exists('review/zone_board.jpg'):
        display(Image('review/zone_board.jpg'))
    if n > 0:
        m = kept[['lon', 'lat', 'score', 'coh', 'pgate']].sort_values('score', ascending=False).reset_index(drop=True)
        m.insert(0, 'id', range(1, len(m) + 1))
        show = m.copy()
        show['map'] = show.apply(lambda r: '<a href="https://www.google.com/maps?q=%s,%s" target="_blank">view</a>' % (r.lat, r.lon), axis=1)
        display(HTML(show.to_html(escape=False, index=False)))
        m.to_csv('detected_mounds.csv', index=False)
        try:
            from google.colab import files
            files.download('detected_mounds.csv')
        except Exception:
            print("detected_mounds.csv saved; see the Files panel on the left.")